# NOTEBOOK : 03_preprocessing.ipynb
# ROLE     : Standardize team names, parse dates, clean types, handle missing values, and export clean_matches.csv.
# INPUT    : ../../data/processed/all_matches_raw.csv
# OUTPUT   : ../artifacts/team_mapping.json, ../../data/processed/clean_matches.csv

In [1]:
from pathlib import Path
import json
import pandas as pd

INPUT_FILE = Path("../../data/processed/all_matches_raw.csv")
ARTIFACTS_DIR = Path("../artifacts")
MAPPING_FILE = ARTIFACTS_DIR / "team_mapping.json"
OUTPUT_FILE = Path("../../data/processed/clean_matches.csv")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Canonical official team names -> covers Match CSVs, FPL Gameweeks, and Understat
TEAM_MAPPING = {
    "Arsenal": "Arsenal",
    "Aston Villa": "Aston Villa",
    "Aston_Villa": "Aston Villa",
    "Bournemouth": "Bournemouth",
    "Brentford": "Brentford",
    "Brighton": "Brighton",
    "Burnley": "Burnley",
    "Chelsea": "Chelsea",
    "Coventry": "Coventry",
    "Crystal Palace": "Crystal Palace",
    "Crystal_Palace": "Crystal Palace",
    "Everton": "Everton",
    "Fulham": "Fulham",
    "Hull": "Hull",
    "Ipswich": "Ipswich",
    "Leeds": "Leeds",
    "Leicester": "Leicester",
    "Liverpool": "Liverpool",
    "Luton": "Luton",
    "Man City": "Man City",
    "Manchester_City": "Man City",
    "Man United": "Man United",
    "Man Utd": "Man United",
    "Manchester_United": "Man United",
    "Newcastle": "Newcastle",
    "Newcastle_United": "Newcastle",
    "Norwich": "Norwich",
    "Nott'm Forest": "Nottingham Forest",
    "Nottingham Forest": "Nottingham Forest",
    "Sheffield United": "Sheffield United",
    "Sheffield Utd": "Sheffield United",
    "Sheffield_United": "Sheffield United",
    "Southampton": "Southampton",
    "Spurs": "Tottenham",
    "Sunderland": "Sunderland",
    "Tottenham": "Tottenham",
    "Watford": "Watford",
    "West Brom": "West Brom",
    "West_Bromwich_Albion": "West Brom",
    "West Ham": "West Ham",
    "West_Ham": "West Ham",
    "Wolves": "Wolves",
    "Wolverhampton_Wanderers": "Wolves",
}

with open(MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(TEAM_MAPPING, f, indent=2, sort_keys=True)

print(f"Saved team mapping ({len(TEAM_MAPPING)} entries) to {MAPPING_FILE}")

Saved team mapping (43 entries) to ..\artifacts\team_mapping.json


In [2]:
df = pd.read_csv(INPUT_FILE)

# Keep core match columns (drop Understat xG/xGA if any; keep bookmaker odds out of core features)
CORE_COLS = [
    "season", "Match_Date", "Home_Team", "Away_Team",
    "Full_Time_Home_Goals", "Full_Time_Away_Goals", "Full_Time_Result",
    "Home_Shots", "Away_Shots", "Home_Shots_On_Target", "Away_Shots_On_Target",
    "Home_Corners", "Away_Corners", "Home_Yellow_Cards", "Away_Yellow_Cards",
    "Home_Red_Cards", "Away_Red_Cards",
]

clean_df = df[CORE_COLS].copy()

# Strip whitespace on string columns
str_cols = ["season", "Match_Date", "Home_Team", "Away_Team", "Full_Time_Result"]
for col in str_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Map team names
clean_df["Home_Team"] = clean_df["Home_Team"].replace(TEAM_MAPPING)
clean_df["Away_Team"] = clean_df["Away_Team"].replace(TEAM_MAPPING)

# Verify no unmapped teams remain
all_mapped_targets = set(TEAM_MAPPING.values())
unmapped_home = set(clean_df["Home_Team"]) - all_mapped_targets
unmapped_away = set(clean_df["Away_Team"]) - all_mapped_targets
assert not unmapped_home and not unmapped_away, f"Unmapped teams found: {unmapped_home | unmapped_away}"
print("All Home_Team and Away_Team values successfully mapped.")

All Home_Team and Away_Team values successfully mapped.


In [3]:
# Parse Match_Date (DD/MM/YYYY -> YYYY-MM-DD)
clean_df["Match_Date"] = pd.to_datetime(clean_df["Match_Date"], dayfirst=True, errors="coerce")
assert clean_df["Match_Date"].isna().sum() == 0, "Some match dates failed to parse!"

# Ensure integer types for numeric stats
int_cols = [
    "Full_Time_Home_Goals", "Full_Time_Away_Goals",
    "Home_Shots", "Away_Shots", "Home_Shots_On_Target", "Away_Shots_On_Target",
    "Home_Corners", "Away_Corners", "Home_Yellow_Cards", "Away_Yellow_Cards",
    "Home_Red_Cards", "Away_Red_Cards",
]
for col in int_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce").astype("Int64")

# Remove void/incomplete rows (if any) and duplicate matches
before_rows = len(clean_df)
clean_df = clean_df.dropna(subset=["Match_Date", "Home_Team", "Away_Team", "Full_Time_Result"])
clean_df = clean_df.drop_duplicates(subset=["Match_Date", "Home_Team", "Away_Team"])
print(f"Removed {before_rows - len(clean_df)} void/duplicate rows.")

# Sort chronologically
clean_df = clean_df.sort_values(["Match_Date", "Home_Team"]).reset_index(drop=True)
clean_df["Match_Date"] = clean_df["Match_Date"].dt.strftime("%Y-%m-%d")

# Assert 20 unique teams per complete season (380 matches)
for season, grp in clean_df.groupby("season"):
    teams_in_season = set(grp["Home_Team"]) | set(grp["Away_Team"])
    if len(grp) == 380:
        assert len(teams_in_season) == 20, f"{season} has {len(teams_in_season)} teams instead of 20!"
    print(f"  {season}: {len(grp)} matches | {len(teams_in_season)} unique teams")

Removed 0 void/duplicate rows.
  2020-21: 380 matches | 20 unique teams
  2021-22: 380 matches | 20 unique teams
  2022-23: 380 matches | 20 unique teams
  2023-24: 380 matches | 20 unique teams
  2024-25: 380 matches | 20 unique teams
  2025-26: 380 matches | 20 unique teams
  2026-27: 70 matches | 20 unique teams


In [4]:
clean_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved clean matches to {OUTPUT_FILE} ({len(clean_df)} rows, {clean_df.shape[1]} columns)")
clean_df.head()

Saved clean matches to ..\..\data\processed\clean_matches.csv (2350 rows, 17 columns)


,season,Match_Date,Home_Team,Away_Team,Full_Time_Home_Goals,Full_Time_Away_Goals,Full_Time_Result,Home_Shots,Away_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards
0,2020-21,2020-09-12,Crystal Palace,Southampton,1,0,H,5,9,3,5,7,3,2,1,0,0
1,2020-21,2020-09-12,Fulham,Arsenal,0,3,A,5,13,2,6,2,3,2,2,0,0
2,2020-21,2020-09-12,Liverpool,Leeds,4,3,H,22,6,6,3,9,0,1,0,0,0
3,2020-21,2020-09-12,West Ham,Newcastle,0,2,A,15,15,3,2,8,7,2,2,0,0
4,2020-21,2020-09-13,Tottenham,Everton,0,1,A,9,15,5,4,5,3,1,0,0,0
